# Class 14: Word Embeddings
### Topics: Word2Vec, GloVe, Cosine Similarity
### Practical: Building a Semantic Search Engine

This notebook demonstrates:
- Training Word2Vec
- Using Cosine Similarity
- Loading GloVe embeddings
- Building a Semantic Search Engine


## 1. Install and Import Required Libraries

In [ ]:
!pip install gensim nltk scikit-learn

In [1]:
import nltk
nltk.download('punkt')

from nltk.tokenize import word_tokenize
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


[nltk_data] Downloading package punkt to C:\Users\Waleed
[nltk_data]     Hassan\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 2. Training a Word2Vec Model

We create a small corpus and train Word2Vec using CBOW (default).

In [6]:
sentences = [
    "I love natural language processing",
    "I Like machine learning and deep learning",
    "Word embeddings capture semantic meaning",
    "Word2Vec and GloVe are popular embedding models",
    "Machine learning enables semantic search",
    "Cosine similarity measures vector closeness"
]

tokenized_sentences = [word_tokenize(sentence.lower()) for sentence in sentences]

model = Word2Vec(
    sentences=tokenized_sentences,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

model.wv.most_similar("cosine")

[('embedding', 0.16695010662078857),
 ('similarity', 0.13891161978244781),
 ('measures', 0.13146455585956573),
 ('word', 0.09771843254566193),
 ('meaning', 0.07171798497438431),
 ('vector', 0.06410502642393112),
 ('learning', 0.06058300659060478),
 ('word2vec', 0.047767311334609985),
 ('are', 0.044107817113399506),
 ('love', 0.04272579774260521)]

## 3. Cosine Similarity

Cosine similarity measures the angle between two vectors:

Cosine(A,B) = (A·B) / (||A|| ||B||)


In [9]:
vec1 = model.wv['cosine']
vec2 = model.wv['embeddings']

similarity = cosine_similarity([vec1], [vec2])
print("Cosine Similarity:", similarity[0][0])

Cosine Similarity: -0.12686111


## 4. GloVe Embeddings

Download and load pretrained GloVe embeddings.

In [18]:
import gensim.downloader as api

print("Loading smaller pretrained embeddings (glove-wiki-gigaword-50)...")
glove = api.load("glove-wiki-gigaword-50")

glove["king"][:5]

Loading smaller pretrained embeddings (glove-wiki-gigaword-50)...
[==================================================] 100.0% 66.0/66.0MB downloaded


array([ 0.50451 ,  0.68607 , -0.59517 , -0.022801,  0.60046 ],
      dtype=float32)

### Analogy Example
king - man + woman ≈ queen

In [21]:
king = glove["king"]
man = glove["man"]
woman = glove["woman"]

result = king - man + woman
excluded = {"king", "man", "woman"}

candidates = glove.similar_by_vector(result, topn=10)
closest_word = next(word for word, _ in candidates if word not in excluded)
closest_word

'queen'

## 5. Building a Semantic Search Engine

We represent each document as the average of its word embeddings.

In [22]:
documents = [
    "Deep learning improves natural language understanding",
    "Word embeddings represent words as dense vectors",
    "Semantic search uses vector similarity",
    "Machine learning powers recommendation systems",
    "Cosine similarity measures distance between vectors"
]

def document_vector(doc):
    words = word_tokenize(doc.lower())
    vectors = [model.wv[word] for word in words if word in model.wv]
    return np.mean(vectors, axis=0)

doc_vectors = [document_vector(doc) for doc in documents]

def semantic_search(query, documents, doc_vectors):
    query_vec = document_vector(query)
    similarities = cosine_similarity([query_vec], doc_vectors)[0]
    ranked_docs = sorted(zip(documents, similarities), key=lambda x: x[1], reverse=True)
    return ranked_docs

query = "How to measure similarity between words?"
results = semantic_search(query, documents, doc_vectors)

for doc, score in results:
    print(f"Score: {score:.4f} | Document: {doc}")

Score: 0.5753 | Document: Cosine similarity measures distance between vectors
Score: 0.5597 | Document: Semantic search uses vector similarity
Score: -0.1028 | Document: Deep learning improves natural language understanding
Score: -0.1672 | Document: Machine learning powers recommendation systems
Score: -0.1948 | Document: Word embeddings represent words as dense vectors
